# LeetCode 405 — Convert a Number to Hexadecimal bằng Python

**Link đề:** https://leetcode.com/problems/convert-a-number-to-hexadecimal/
**Độ khó:** Easy · **Chủ đề:** Math, String, Bit Manipulation

```python
class Solution:
    def toHex(self, num: int) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **hệ thập lục phân** (hexadecimal, cơ số 16) và cách đổi số dương sang hex bằng chia lấy dư.
2. Hiểu **bù hai** (two's complement) — cách máy tính lưu số âm 32-bit — và vì sao `-1` thành `"ffffffff"`.
3. Nắm các toán tử bit `&`, `>>`, mặt nạ (mask) `0xffffffff`, và điểm khác biệt của số nguyên Python (không giới hạn bit).
4. Viết 4 lời giải: chia 16 sau khi cộng `2**32` → lấy 4 bit thấp bằng `& 15` + `>> 4` → duyệt 8 nhóm 4 bit từ cao xuống → (đối chiếu) hàm có sẵn.
5. Trace từng bước, chứng minh tính đúng, Time/Space, lỗi hay gặp và biến thể (đổi sang cơ số bất kỳ).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho số nguyên 32-bit `num`, trả về **chuỗi biểu diễn thập lục phân** của nó. Với số âm dùng phương pháp **bù hai** (two's complement).

- Mọi chữ cái trong kết quả phải là **chữ thường** (`a`..`f`).
- **Không có số 0 thừa ở đầu**, trừ chính số `0` (kết quả `"0"`).
- **Không được** dùng hàm thư viện đổi trực tiếp (như `hex()`, `format(num, 'x')`).

| `num` | Output | Ghi chú |
|---:|---|---|
| `26` | `"1a"` | 26 = 1·16 + 10, chữ số 10 viết là `a` |
| `-1` | `"ffffffff"` | bù hai 32-bit của −1 là 32 bit 1 |
| `0` | `"0"` | trường hợp đặc biệt |

**Constraints:** `-2^31 <= num <= 2^31 - 1`.

Câu hỏi đặt ra:
- Hex là gì, và đổi số dương sang hex thế nào?
- Số âm "trông" như thế nào trong 32 bit?
- Trong Python số nguyên không có giới hạn bit — vậy làm sao "giả lập" 32 bit?

## 2. Nền tảng: hệ cơ số 16

Hệ thập phân có 10 chữ số `0..9`. Hệ thập lục phân có **16** chữ số: `0..9` rồi `a=10, b=11, c=12, d=13, e=14, f=15`.

Mỗi vị trí mạnh gấp 16 lần vị trí bên phải:

`"1a"` = `1·16 + 10` = `26`; `"ff"` = `15·16 + 15` = `255`; `"100"` = `1·16² = 256`.

Ta lưu bảng chữ số trong một chuỗi và **truy cập theo chỉ số** (indexing): `digits[10]` là `'a'`.

In [1]:
digits = "0123456789abcdef"
print("len:", len(digits))
print("digits[0] =", digits[0], "| digits[10] =", digits[10], "| digits[15] =", digits[15])

# Đọc "1a" về số thập phân: tìm vị trí của mỗi ký tự trong bảng
value = 0
for ch in "1a":
    value = value * 16 + digits.index(ch)
print('"1a" ->', value)

len: 16
digits[0] = 0 | digits[10] = a | digits[15] = f
"1a" -> 26


### 2.1. Đổi số dương sang hex: chia lấy dư cho 16

Giống đổi sang nhị phân: lặp lại
- `num % 16` → chữ số **cuối** (hàng đơn vị),
- `num // 16` → bỏ chữ số cuối.

Các chữ số thu được theo thứ tự **ngược** (từ phải qua trái), nên cuối cùng phải đảo lại.

Ví dụ `26`: `26 % 16 = 10 → 'a'`, `26 // 16 = 1`; `1 % 16 = 1 → '1'`, `1 // 16 = 0` dừng. Thu được `['a','1']` → đảo → `"1a"`.

In [2]:
num = 26
print(num % 16, num // 16)     # 10 1
num = 1
print(num % 16, num // 16)     # 1 0

# Khác biệt / và //
print(26 / 16, 26 // 16)

10 1
1 0
1.625 1


### 2.2. Ghép chuỗi: `list` + `reversed` + `"".join`

Nối chuỗi bằng `+` trong vòng lặp tạo chuỗi mới mỗi lần. Cách quen thuộc hơn: gom ký tự vào `list`, cuối cùng `"".join(...)`. `reversed(lst)` duyệt list từ cuối về đầu, `lst[::-1]` tạo list đảo.

In [3]:
parts = ['a', '1']
print(list(reversed(parts)))
print("".join(reversed(parts)))
print("".join(parts[::-1]))
print("-".join(["x", "y", "z"]))   # ký tự nối có thể khác rỗng

['1', 'a']
1a
1a
x-y-z


### 2.3. Bù hai (two's complement) với 32 bit

Máy tính lưu số nguyên 32-bit có dấu trong 32 ô bit. Số âm `x` được lưu bằng **mẫu bit của `x + 2^32`**.

- `-1 + 2^32 = 4294967295 = 0xffffffff` → 32 bit 1 → `"ffffffff"`.
- `-2 + 2^32 = 4294967294 = 0xfffffffe` → `"fffffffe"`.
- `-2^31 + 2^32 = 2^31 = 0x80000000`.

Nói cách khác: **bù hai = cộng `2^32` cho số âm** rồi đổi hex như số dương. Một số 32 bit có đúng `32 / 4 = 8` chữ số hex.

In [4]:
for x in [-1, -2, -26, -2**31]:
    u = x + 2**32
    print(f"{x:>12} -> {u:>11} (bit: {bin(u)[2:]:>32})")

          -1 ->  4294967295 (bit: 11111111111111111111111111111111)
          -2 ->  4294967294 (bit: 11111111111111111111111111111110)
         -26 ->  4294967270 (bit: 11111111111111111111111111100110)
 -2147483648 ->  2147483648 (bit: 10000000000000000000000000000000)


### 2.4. Toán tử bit `&` và `>>`, mặt nạ `0xffffffff`

- `a & b` (AND từng bit): bit kết quả là 1 khi **cả hai** bit là 1.
- `x & 15` (15 = `0b1111`) giữ lại **4 bit thấp nhất** = chữ số hex cuối = `x % 16` (với `x >= 0`).
- `x >> 4` dịch phải 4 bit = bỏ chữ số hex cuối = `x // 16`.
- `x & 0xffffffff` giữ lại **32 bit thấp**. Với số âm trong Python, kết quả đúng bằng `x + 2^32` → chính là bù hai!

**Lưu ý Python:** số nguyên Python không giới hạn bit; số âm được coi như có **vô hạn** bit 1 ở bên trái. Vì vậy `-1 >> 4` vẫn là `-1` mãi mãi → vòng `while num != 0` sẽ **lặp vô hạn** nếu không xử lý.

In [5]:
x = 26
print("26 & 15 =", x & 15, "| 26 % 16 =", x % 16)
print("26 >> 4 =", x >> 4, "| 26 // 16 =", x // 16)
print("-1 & 0xffffffff =", -1 & 0xffffffff, "=", 2**32 - 1)
print("-26 & 0xffffffff =", -26 & 0xffffffff, "=", -26 + 2**32)
print("-1 >> 4 =", -1 >> 4, " (không về 0!)")
print("0xffffffff =", 0xffffffff, "| hex literal 0x1a =", 0x1a)

26 & 15 = 10 | 26 % 16 = 10
26 >> 4 = 1 | 26 // 16 = 1
-1 & 0xffffffff = 4294967295 = 4294967295
-26 & 0xffffffff = 4294967270 = 4294967270
-1 >> 4 = -1  (không về 0!)
0xffffffff = 4294967295 | hex literal 0x1a = 26


## 3. Cách trực giác (baseline) — dùng hàm có sẵn (chỉ để đối chiếu)

Python có `format(x, 'x')` và `hex(x)`. Đề **cấm** dùng chúng trong bài nộp, nhưng chúng rất hữu ích làm **oracle** (đáp án chuẩn) khi test lời giải tự viết.

Chú ý `hex(-1)` cho `'-0x1'` — **không** phải bù hai. Phải đổi sang dạng không dấu trước.

In [6]:
def to_hex_builtin(num: int) -> str:
    # CHỈ dùng để kiểm tra, không nộp
    return format(num & 0xffffffff, "x")

print(hex(26), hex(-1))
for x in [26, -1, 0, 255, -2**31, 2**31 - 1]:
    print(x, "->", to_hex_builtin(x))

0x1a -0x1
26 -> 1a
-1 -> ffffffff
0 -> 0
255 -> ff
-2147483648 -> 80000000
2147483647 -> 7fffffff


## 4. Nút thắt và insight

**Nút thắt 1 — số âm:** chia lấy dư trực tiếp trên số âm không cho bù hai (`-1 % 16 = 15`, `-1 // 16 = -1` → lặp vô hạn).

**Nút thắt 2 — số 0:** vòng `while num > 0` không chạy lần nào → chuỗi rỗng `""` thay vì `"0"`.

**Insight:**
1. Đổi số âm thành số không dấu tương ứng: `num += 2**32` (hoặc `num &= 0xffffffff`). Sau bước này mọi số đều trong `[0, 2^32)`, chỉ cần đổi hex "như số dương".
2. Xử lý `num == 0` riêng ngay đầu.
3. Một chữ số hex = 4 bit → có thể dùng `& 15` và `>> 4` thay `% 16` và `// 16`.

In [7]:
print("-1 % 16 =", -1 % 16, "| -1 // 16 =", -1 // 16, "-> không bao giờ về 0")
num = -1
num += 2**32
print("sau khi cộng 2^32:", num, "-> giờ chia 16 sẽ dừng")

-1 % 16 = 15 | -1 // 16 = -1 -> không bao giờ về 0
sau khi cộng 2^32: 4294967295 -> giờ chia 16 sẽ dừng


## 5. Lời giải chính — chia 16 trên số không dấu

**Đoạn 1 — số 0:**
```python
if num == 0:
    return "0"
```
**Đoạn 2 — đổi số âm sang bù hai:**
```python
if num < 0:
    num += 2 ** 32
```
**Đoạn 3 — bóc từng chữ số hex từ phải sang trái:**
```python
out = []
while num > 0:
    out.append(digits[num % 16])
    num //= 16
```
**Đoạn 4 — đảo và ghép:** `return "".join(reversed(out))`.

Không có số 0 thừa ở đầu vì vòng lặp dừng ngay khi `num` hết chữ số có nghĩa.

In [8]:
HEX_DIGITS = "0123456789abcdef"

def to_hex_div(num: int) -> str:
    if num == 0:
        return "0"
    if num < 0:
        num += 2 ** 32
    out = []
    while num > 0:
        out.append(HEX_DIGITS[num % 16])
        num //= 16
    return "".join(reversed(out))


for x in [26, -1, 0, 16, 255, -2**31, 2**31 - 1]:
    print(f"{x:>12} -> {to_hex_div(x)}")

          26 -> 1a
          -1 -> ffffffff
           0 -> 0
          16 -> 10
         255 -> ff
 -2147483648 -> 80000000
  2147483647 -> 7fffffff


## 6. TRACE từng bước

Bảng trace cho `num = 26`:

| Vòng | num trước | num % 16 | ký tự | out sau | num sau (//16) |
|:---:|---:|---:|:---:|---|---:|
| 1 | 26 | 10 | `a` | `['a']` | 1 |
| 2 | 1 | 1 | `1` | `['a','1']` | 0 |

Dừng vì `num = 0`; đảo → `"1a"`.

Với `num = -26`: cộng `2^32` → `4294967270`, rồi bóc 8 chữ số: `6, e, f, f, f, f, f, f` → `"ffffffe6"`.

In [9]:
def trace_to_hex(num: int) -> str:
    print(f"Input: {num}")
    if num == 0:
        print("  num == 0 -> trả '0'")
        return "0"
    if num < 0:
        num += 2 ** 32
        print(f"  số âm -> cộng 2^32 -> {num}")
    out = []
    step = 1
    while num > 0:
        d = num % 16
        out.append(HEX_DIGITS[d])
        print(f"  vòng {step}: num={num:>10}, num%16={d:>2} -> '{HEX_DIGITS[d]}', out={out}, num//16={num // 16}")
        num //= 16
        step += 1
    res = "".join(reversed(out))
    print(f"  đảo out -> '{res}'")
    return res

trace_to_hex(26)
print()
trace_to_hex(-26)

Input: 26
  vòng 1: num=        26, num%16=10 -> 'a', out=['a'], num//16=1
  vòng 2: num=         1, num%16= 1 -> '1', out=['a', '1'], num//16=0
  đảo out -> '1a'

Input: -26
  số âm -> cộng 2^32 -> 4294967270
  vòng 1: num=4294967270, num%16= 6 -> '6', out=['6'], num//16=268435454
  vòng 2: num= 268435454, num%16=14 -> 'e', out=['6', 'e'], num//16=16777215
  vòng 3: num=  16777215, num%16=15 -> 'f', out=['6', 'e', 'f'], num//16=1048575
  vòng 4: num=   1048575, num%16=15 -> 'f', out=['6', 'e', 'f', 'f'], num//16=65535
  vòng 5: num=     65535, num%16=15 -> 'f', out=['6', 'e', 'f', 'f', 'f'], num//16=4095
  vòng 6: num=      4095, num%16=15 -> 'f', out=['6', 'e', 'f', 'f', 'f', 'f'], num//16=255
  vòng 7: num=       255, num%16=15 -> 'f', out=['6', 'e', 'f', 'f', 'f', 'f', 'f'], num//16=15
  vòng 8: num=        15, num%16=15 -> 'f', out=['6', 'e', 'f', 'f', 'f', 'f', 'f', 'f'], num//16=0
  đảo out -> 'ffffffe6'


'ffffffe6'

## 7. Các lời giải khác

### 7.1. Cách 3 — thao tác bit: `& 15` và `>> 4`, có mặt nạ

Thay `% 16` bằng `& 0xf`, `// 16` bằng `>> 4`. Trước đó dùng `num &= 0xffffffff` để đưa số âm về 32 bit không dấu (tương đương cộng `2^32`).

In [10]:
def to_hex_bits(num: int) -> str:
    if num == 0:
        return "0"
    num &= 0xffffffff
    out = []
    while num:
        out.append(HEX_DIGITS[num & 0xf])
        num >>= 4
    return "".join(reversed(out))

for x in [26, -1, 0, -26, 2**31 - 1]:
    print(f"{x:>12} -> {to_hex_bits(x)}")

          26 -> 1a
          -1 -> ffffffff
           0 -> 0
         -26 -> ffffffe6
  2147483647 -> 7fffffff


### 7.2. Cách 4 — tối đa 8 vòng, không cần mặt nạ (kiểu C/Java)

Ý tưởng hay gặp trong ngôn ngữ có số nguyên 32-bit: chỉ lặp **tối đa 8 lần** (vì 32 bit = 8 chữ số hex). Lấy `num & 15` cho chữ số, `num >>= 4`. Dừng khi `num == 0` **hoặc** đã đủ 8 chữ số.

Trong Python, `-1 >> 4` luôn là `-1`, nhưng nhờ giới hạn 8 vòng nên vẫn dừng; và `-1 & 15 = 15` đúng là `f` vì phép `&` trên số âm Python vẫn lấy đúng bit bù hai.

In [11]:
def to_hex_8_rounds(num: int) -> str:
    if num == 0:
        return "0"
    out = []
    while num != 0 and len(out) < 8:
        out.append(HEX_DIGITS[num & 15])
        num >>= 4
    return "".join(reversed(out))

for x in [26, -1, -26, -2**31, 2**31 - 1]:
    print(f"{x:>12} -> {to_hex_8_rounds(x)}")

          26 -> 1a
          -1 -> ffffffff
         -26 -> ffffffe6
 -2147483648 -> 80000000
  2147483647 -> 7fffffff


### 7.3. Cách 5 — duyệt 8 nhóm 4 bit từ **cao xuống thấp**

Không cần đảo chuỗi: đi từ nhóm bit 28..31 xuống 0..3, lấy `(num >> shift) & 15`. Bỏ qua các chữ số 0 **đứng đầu** (khi `out` còn rỗng).

In [12]:
def to_hex_high_to_low(num: int) -> str:
    num &= 0xffffffff
    out = []
    for shift in range(28, -1, -4):          # 28, 24, ..., 4, 0
        d = (num >> shift) & 15
        if d == 0 and not out:               # số 0 dẫn đầu -> bỏ
            continue
        out.append(HEX_DIGITS[d])
    return "".join(out) or "0"               # rỗng nghĩa là num == 0

print(list(range(28, -1, -4)))
for x in [26, -1, 0, 256, -2**31]:
    print(f"{x:>12} -> {to_hex_high_to_low(x)}")

[28, 24, 20, 16, 12, 8, 4, 0]
          26 -> 1a
          -1 -> ffffffff
           0 -> 0
         256 -> 100
 -2147483648 -> 80000000


Cú pháp mới: `range(start, stop, step)` với `step` âm đi **lùi**; `stop` không được lấy (nên dùng `-1` để có số `0`). Và `A or B` trả `B` nếu `A` là "rỗng/False" — mẹo gọn để đổi `""` thành `"0"`.

In [13]:
print(list(range(10, -1, -5)), list(range(10, 0, -5)))
print(repr("" or "0"), repr("1a" or "0"))

[10, 5, 0] [10, 5]
'0' '1a'


## 8. Code nộp LeetCode

Bản chính dùng bit (`& 15`, `>> 4`); bản phụ dùng chia 16. Cả hai đúng chữ ký `toHex(self, num) -> str`.

In [14]:
class Solution:
    def toHex(self, num: int) -> str:
        if num == 0:
            return "0"
        digits = "0123456789abcdef"
        num &= 0xffffffff          # bù hai 32-bit cho số âm
        out = []
        while num:
            out.append(digits[num & 15])
            num >>= 4
        return "".join(reversed(out))


sol = Solution()
print(sol.toHex(26))   # 1a
print(sol.toHex(-1))   # ffffffff

1a
ffffffff


In [15]:
class SolutionDivide:
    def toHex(self, num: int) -> str:
        if num == 0:
            return "0"
        if num < 0:
            num += 2 ** 32
        digits = "0123456789abcdef"
        out = []
        while num > 0:
            out.append(digits[num % 16])
            num //= 16
        return "".join(out[::-1])

print(SolutionDivide().toHex(26), SolutionDivide().toHex(-1))

1a ffffffff


## 9. Vì sao đúng? (invariant)

Gọi `N` là giá trị không dấu (`num` sau khi cộng `2^32` nếu âm), `0 < N < 2^32`.

**Bất biến của vòng lặp:** sau mỗi vòng, `N = num_hiện_tại · 16^k + giá_trị(out đảo ngược)`, với `k = len(out)`.
- Ban đầu `k = 0`, `out` rỗng → đúng.
- Mỗi vòng: `num = 16·(num // 16) + num % 16` → chuyển chữ số `num % 16` vào vị trí `16^k`, `num` giảm thành `num // 16` → bất biến giữ nguyên.
- Kết thúc `num = 0` → `N = giá_trị(out đảo)`. Chữ số cuối cùng được thêm là chữ số khác 0 (vì `num > 0` và `num < 16` ở vòng cuối) → **không có 0 thừa ở đầu**.

**Tính đúng của bù hai:** với `-2^31 <= num < 0`, mẫu 32 bit của `num` chính là biểu diễn nhị phân của `num + 2^32`. Và `num & 0xffffffff == num + 2^32` với mọi `num` trong khoảng đó.

In [16]:
ok = all((x & 0xffffffff) == x + 2**32 for x in range(-2**31, 0, 9973))
ok = ok and (-2**31 & 0xffffffff) == 2**31 and (-1 & 0xffffffff) == 2**32 - 1
print("num & 0xffffffff == num + 2^32 với số âm 32-bit:", ok)

num & 0xffffffff == num + 2^32 với số âm 32-bit: True


## 10. Độ phức tạp

Số chữ số hex tối đa là 8 (32 bit / 4). Gọi `d` là số chữ số hex của kết quả (`d <= 8`).

| Cách | Time | Space |
|---|:---:|:---:|
| Hàm có sẵn (oracle) | `O(d)` | `O(d)` |
| Chia 16 + cộng `2^32` | `O(d)` | `O(d)` cho chuỗi kết quả |
| `& 15`, `>> 4` + mặt nạ | `O(d)` | `O(d)` |
| Tối đa 8 vòng | `O(d)` | `O(d)` |
| Cao xuống thấp | `O(8) = O(1)` (luôn 8 vòng) | `O(d)` |

Vì `d <= 8` nên mọi cách đều là **O(1)** theo nghĩa hằng số. Nếu tổng quát lên số `B` bit thì là `O(B / 4)`.

## 11. Bộ test

- Ví dụ đề: `26 → "1a"`, `-1 → "ffffffff"`.
- Biên: `0`, `1`, `15 → "f"`, `16 → "10"`, `2^31-1 → "7fffffff"`, `-2^31 → "80000000"`.
- Phản ví dụ "số 0 thừa": `256 → "100"` (có 0 ở giữa/cuối nhưng không ở đầu).
- Chữ thường: không được ra `"1A"`.
- Đối chiếu tất cả các cách với oracle trên nhiều số ngẫu nhiên.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<22} -> {actual!r}")

sol = Solution()
cases = [
    ("đề 1", 26, "1a"), ("đề 2", -1, "ffffffff"), ("không", 0, "0"),
    ("một", 1, "1"), ("mười lăm", 15, "f"), ("mười sáu", 16, "10"),
    ("256 có số 0 giữa", 256, "100"), ("max int", 2**31 - 1, "7fffffff"),
    ("min int", -2**31, "80000000"), ("-26", -26, "ffffffe6"),
]
for name, x, exp in cases:
    check(name, sol.toHex(x), exp)

PASS  đề 1                   -> '1a'
PASS  đề 2                   -> 'ffffffff'
PASS  không                  -> '0'
PASS  một                    -> '1'
PASS  mười lăm               -> 'f'
PASS  mười sáu               -> '10'
PASS  256 có số 0 giữa       -> '100'
PASS  max int                -> '7fffffff'
PASS  min int                -> '80000000'
PASS  -26                    -> 'ffffffe6'


In [18]:
import random
rng = random.Random(405)
samples = [rng.randint(-2**31, 2**31 - 1) for _ in range(3000)] + list(range(-300, 300))
funcs = [to_hex_div, to_hex_bits, to_hex_8_rounds, to_hex_high_to_low,
         Solution().toHex, SolutionDivide().toHex]
for x in samples:
    exp = to_hex_builtin(x)
    for f in funcs:
        assert f(x) == exp, (f.__name__, x)
print(f"{len(samples)} số: 6 cách khớp oracle -> PASS")

3600 số: 6 cách khớp oracle -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên `num == 0`
```python
out = []
while num > 0: ...
return "".join(reversed(out))     # num = 0 -> ""
```
**Sửa:** `if num == 0: return "0"` (hoặc `... or "0"`).

### Lỗi 2: Dùng `while num != 0` với số âm mà không mặt nạ
`-1 >> 4 == -1` → lặp vô hạn. **Sửa:** `num &= 0xffffffff` trước, hoặc giới hạn 8 vòng.

In [19]:
def wrong_zero(num):
    out = []
    while num > 0:
        out.append(HEX_DIGITS[num % 16])
        num //= 16
    return "".join(reversed(out))

print("lỗi 1 với 0:", repr(wrong_zero(0)))

# Lỗi 2: mô phỏng an toàn — chỉ cho chạy 12 vòng để thấy num không về 0
num = -1
for i in range(12):
    num >>= 4
print("sau 12 lần >> 4, -1 vẫn là:", num)

lỗi 1 với 0: ''
sau 12 lần >> 4, -1 vẫn là: -1


### Lỗi 3: Dùng `hex()` cho số âm
`hex(-1) == '-0x1'` — vừa có tiền tố `0x`, vừa có dấu `-`, không phải bù hai. Đề cũng cấm dùng hàm này.

### Lỗi 4: Quên đảo chuỗi
Bóc chữ số từ phải sang trái nên `out = ['a','1']`; ghép thẳng ra `"a1"` sai.

### Lỗi 5: Chữ hoa
Dùng bảng `"0123456789ABCDEF"` → `"1A"` sai yêu cầu chữ thường.

In [20]:
def wrong_no_reverse(num):
    num &= 0xffffffff
    out = []
    while num:
        out.append(HEX_DIGITS[num & 15])
        num >>= 4
    return "".join(out)

print("hex(-1) =", hex(-1))
print("quên đảo, 26 ->", wrong_no_reverse(26), "(đúng: 1a)")
print("chữ hoa  ->", "0123456789ABCDEF"[10], "(đúng: a)")

hex(-1) = -0x1
quên đảo, 26 -> a1 (đúng: 1a)
chữ hoa  -> A (đúng: a)


## 13. Mẹo nhận dạng pattern

- "Đổi sang cơ số `b`" → lặp `x % b` (chữ số cuối), `x //= b`, rồi đảo.
- Cơ số là lũy thừa của 2 (2, 8, 16) → thay bằng `& (b-1)` và `>> log2(b)`.
- "32-bit", "two's complement", "unsigned" trong Python → nghĩ ngay tới `x & 0xffffffff` (hoặc `x + 2**32`).
- Python không tràn số: mọi bài "giả lập 32-bit" phải **tự** mặt nạ.

**Template đổi cơ số:**
```python
def to_base(x, b, digits="0123456789abcdefghijklmnopqrstuvwxyz"):
    if x == 0: return "0"
    out = []
    while x > 0:
        out.append(digits[x % b]); x //= b
    return "".join(reversed(out))
```

## 14. Biến thể

1. **LeetCode 504 — Base 7:** đổi sang cơ số 7, số âm thêm dấu `-` (không dùng bù hai).
2. **LeetCode 168 — Excel Sheet Column Title:** cơ số 26 không có chữ số 0 (dùng `(x-1) % 26`).
3. **Đổi ngược hex → số nguyên có dấu 32-bit.**
4. **Đổi sang nhị phân/bát phân** bù hai.

In [21]:
def to_base(x: int, base: int) -> str:
    digits = "0123456789abcdefghijklmnopqrstuvwxyz"
    if x == 0:
        return "0"
    sign = "-" if x < 0 else ""
    x = abs(x)
    out = []
    while x > 0:
        out.append(digits[x % base])
        x //= base
    return sign + "".join(reversed(out))

def hex_to_int32(s: str) -> int:
    value = 0
    for ch in s:
        value = value * 16 + HEX_DIGITS.index(ch)
    if value >= 2 ** 31:          # bit dấu bật -> số âm
        value -= 2 ** 32
    return value

print("100 base 7:", to_base(100, 7), "| -7 base 7:", to_base(-7, 7))
print("255 base 2:", to_base(255, 2))
print("'ffffffff' ->", hex_to_int32("ffffffff"), "| '1a' ->", hex_to_int32("1a"))

100 base 7: 202 | -7 base 7: -10
255 base 2: 11111111
'ffffffff' -> -1 | '1a' -> 26


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** Đổi `500` sang hex bằng chia 16. Lập bảng `num`, `num % 16`, `num // 16`.

**Bài 2 (đoán output):** `-16 & 0xffffffff` bằng bao nhiêu, và `Solution().toHex(-16)` in gì?

**Bài 3 (điền chỗ trống):** Hoàn thiện `exercise_to_hex`.

In [22]:
def exercise_to_hex(num: int) -> str:
    # TODO 1: num == 0 -> "0"
    # TODO 2: đưa số âm về 32-bit không dấu
    # TODO 3: bóc chữ số bằng & 15 và >> 4
    # TODO 4: đảo và ghép
    pass

# assert exercise_to_hex(26) == "1a"
# assert exercise_to_hex(-1) == "ffffffff"
# assert exercise_to_hex(0) == "0"

**Bài 4 (tự viết):** `exercise_to_binary32(num)` trả chuỗi nhị phân bù hai 32-bit, không số 0 thừa ở đầu (`5 → "101"`, `-1 → "1"*32`, `0 → "0"`).

**Bài 5 (biến thể):** `exercise_hex_to_int32(s)` đổi chuỗi hex (chữ thường, tối đa 8 ký tự) về số nguyên có dấu 32-bit, không dùng `int(s, 16)`.

**Bài 6 (mini challenge):** `exercise_to_hex_padded(num, width)` trả hex bù hai, có thêm số 0 bên trái cho đủ `width` ký tự (nếu dài hơn thì giữ nguyên).

In [23]:
def exercise_to_binary32(num: int) -> str:
    # TODO
    pass

def exercise_hex_to_int32(s: str) -> int:
    # TODO
    pass

def exercise_to_hex_padded(num: int, width: int) -> str:
    # TODO
    pass

# assert exercise_to_binary32(5) == "101"
# assert exercise_hex_to_int32("80000000") == -2**31
# assert exercise_to_hex_padded(26, 4) == "001a"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chia 16 liên tiếp | Ghi dư mỗi bước | 500 = 31·16 + 4, 31 = 1·16 + 15 |
| 2 | `& 0xffffffff` = cộng `2^32` | `2^32 - 16` | Bit thấp nhất nhóm 4 là 0 |
| 3 | Theo mục 8 | `num &= 0xffffffff` | `out.append(digits[num & 15]); num >>= 4` |
| 4 | Như hex nhưng cơ số 2 | `& 1`, `>> 1` | Mặt nạ trước, `or "0"` sau |
| 5 | Đọc trái sang phải `*16 + d` | `digits.index(ch)` | Nếu `>= 2**31` thì trừ `2**32` |
| 6 | Dùng lại `toHex` | Số 0 cần thêm = `width - len(s)` | `"0" * k + s`, `k` không âm |

## 17. Đáp án tham khảo

- **Bài 1:**

| num | num % 16 | ký tự | num // 16 |
|---:|---:|:---:|---:|
| 500 | 4 | `4` | 31 |
| 31 | 15 | `f` | 1 |
| 1 | 1 | `1` | 0 |

Đảo `['4','f','1']` → **`"1f4"`**.

- **Bài 2:** `-16 & 0xffffffff = 4294967280`, hex là **`"fffffff0"`**.

In [24]:
print("Bài 1:", Solution().toHex(500))
print("Bài 2:", -16 & 0xffffffff, Solution().toHex(-16))

Bài 1: 1f4
Bài 2: 4294967280 fffffff0


In [25]:
def answer_to_hex(num: int) -> str:
    if num == 0:
        return "0"
    num &= 0xffffffff
    out = []
    while num:
        out.append(HEX_DIGITS[num & 15])
        num >>= 4
    return "".join(reversed(out))

def answer_to_binary32(num: int) -> str:
    num &= 0xffffffff
    out = []
    while num:
        out.append(str(num & 1))
        num >>= 1
    return "".join(reversed(out)) or "0"

def answer_hex_to_int32(s: str) -> int:
    value = 0
    for ch in s:
        value = value * 16 + HEX_DIGITS.index(ch)
    return value - 2 ** 32 if value >= 2 ** 31 else value

def answer_to_hex_padded(num: int, width: int) -> str:
    s = answer_to_hex(num)
    return "0" * max(0, width - len(s)) + s

assert answer_to_hex(26) == "1a" and answer_to_hex(-1) == "ffffffff" and answer_to_hex(0) == "0"
assert answer_to_binary32(5) == "101" and answer_to_binary32(-1) == "1" * 32 and answer_to_binary32(0) == "0"
assert answer_hex_to_int32("80000000") == -2**31 and answer_hex_to_int32("ffffffff") == -1
assert answer_hex_to_int32("1a") == 26
assert answer_to_hex_padded(26, 4) == "001a" and answer_to_hex_padded(-1, 4) == "ffffffff"
for x in range(-1000, 1000, 7):
    assert answer_hex_to_int32(answer_to_hex(x)) == x
print("Đáp án bài 3–6 PASS (kèm kiểm tra đổi đi đổi lại)")

Đáp án bài 3–6 PASS (kèm kiểm tra đổi đi đổi lại)


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def toHex(self, num: int) -> str:
        if num == 0:
            return "0"
        digits = "0123456789abcdef"
        num &= 0xffffffff               # bù hai 32-bit
        out = []
        while num:
            out.append(digits[num & 15])  # 4 bit thấp = 1 chữ số hex
            num >>= 4
        return "".join(reversed(out))
```

**Checklist:**
- [ ] `num == 0` trả `"0"`.
- [ ] Số âm được đưa về `num + 2^32` (hoặc `& 0xffffffff`) — tránh lặp vô hạn.
- [ ] Đảo thứ tự chữ số trước khi ghép.
- [ ] Chữ thường `a..f`, không tiền tố `0x`, không số 0 thừa ở đầu.
- [ ] Không dùng `hex()` / `format()` trong bài nộp.
- [ ] Time/Space `O(1)` (tối đa 8 chữ số).